## Setup

In [1]:
import pandas as pd
from pathlib import Path

# where the files are located in the root directory
RAW = Path("../data/raw") 
INTERIM = Path("../data/interim")

INTERIM.mkdir(exist_ok=True)

print("Raw data exists:", RAW.exists())
print("Interim folder exists:", INTERIM.exists())

Raw data exists: True
Interim folder exists: True


### 1) Starting with the hospital information:

In [2]:
hospital_info = pd.read_csv(
    RAW / "hospital_general_information.csv",
    dtype=str,
    keep_default_na=False
)


footnotes = pd.read_csv(
    RAW / "footnote_crosswalk.csv",
    dtype=str,
    keep_default_na=False
)



In [3]:
hospital_info_nm = hospital_info[hospital_info["State"] == 'NM'].copy()

print("NM Hospitals ", len(hospital_info_nm))

# for _, hospital in hospital_info_nm.iterrows():
#     print(hospital["Facility ID"], hospital["Facility Name"])


NM Hospitals  45


In [4]:
# basic column cleanup
hospital_info_nm["Facility ID"] = hospital_info_nm["Facility ID"].str.strip()
hospital_info_nm["ZIP Code"] = hospital_info_nm["ZIP Code"].str[:5]

# flag Critical Access hospitals (often excluded from CMS payment-linked programs)
hospital_info_nm["is_critical_access"] = (
    hospital_info_nm["Hospital Type"] == "Critical Access Hospitals"
)
print("Critical Access hospitals:", hospital_info_nm["is_critical_access"].sum())


Critical Access hospitals: 13


In [5]:
display(
    hospital_info_nm[
        ["Facility ID", "Facility Name", "ZIP Code"]
    ].head()
)

# print(hospital_info_nm[["Facility ID", "ZIP Code"]].dtypes)

,Facility ID,Facility Name,ZIP Code
3154,320001,UNM HOSPITAL,87106
3155,320002,CHRISTUS ST VINCENT REGIONAL MEDICAL CENTER,87505
3156,320004,CHRISTUS SOUTHERN NEW MEXICO,88310
3157,320005,SAN JUAN REGIONAL MEDICAL CENTER INC,87401
3158,320006,EASTERN NEW MEXICO MEDICAL CENTER,88201


In [6]:
#outputs: <length of output>      <number of rows>
print(hospital_info_nm["ZIP Code"].str.len().value_counts())
print("Duplicate Facility IDs:", hospital_info_nm["Facility ID"].duplicated().sum())

ZIP Code
5    45
Name: count, dtype: int64
Duplicate Facility IDs: 0


### Now taking a look at the ```Hospital overall rating```

this is what we got from the exploration notebook:
- 16 → "There are too few measures or measure groups reported to calculate a star rating..."
- 19 → "Data are shown only for hospitals that participate in the IQR and OQR programs"

In [7]:
hospital_info_nm[
    ["Hospital overall rating", "Hospital overall rating footnote"]
].value_counts(dropna=False)

Hospital overall rating  Hospital overall rating footnote
Not Available            16                                  17
2                                                            11
3                                                             5
Not Available            19                                   5
1                                                             3
4                                                             3
5                                                             1
Name: count, dtype: int64

Creating a dedicated column for easier traversal:

In [8]:
hospital_info_nm["overall_rating"] = pd.to_numeric(
    hospital_info_nm["Hospital overall rating"], errors="coerce" # coerce if value cannot be converted, replace with NaN
)

# getting the footnotes from the csv
footnote_lookup = footnotes.set_index("Footnote")["Footnote Text"]

hospital_info_nm["overall_rating_missing_reason"] = (
    hospital_info_nm["Hospital overall rating footnote"].replace("", pd.NA)
    .map(footnote_lookup)
)

# every non-blank rating footnote must decode, and a missing rating must have a reason
unmapped_footnotes = hospital_info_nm.loc[
    hospital_info_nm["Hospital overall rating footnote"].ne("")
    & hospital_info_nm["overall_rating_missing_reason"].isna(),
    "Hospital overall rating footnote",
].unique()
assert len(unmapped_footnotes) == 0
assert hospital_info_nm["overall_rating"].isna().equals(
    hospital_info_nm["overall_rating_missing_reason"].notna()
)


In [9]:
print(
    hospital_info_nm["overall_rating"]
    .value_counts(dropna=False)
    .sort_index()
)

display(
    hospital_info_nm[
        ["Facility Name", "overall_rating", "overall_rating_missing_reason"]
    ]
)

overall_rating
1.0     3
2.0    11
3.0     5
4.0     3
5.0     1
NaN    22
Name: count, dtype: int64


,Facility Name,overall_rating,overall_rating_missing_reason
3154,UNM HOSPITAL,2.0,NaN
3155,CHRISTUS ST VINCENT REGIONAL MEDICAL CENTER,4.0,NaN
3156,CHRISTUS SOUTHERN NEW MEXICO,3.0,NaN
3157,SAN JUAN REGIONAL MEDICAL CENTER INC,2.0,NaN
3158,EASTERN NEW MEXICO MEDICAL CENTER,1.0,NaN
3159,LOVELACE MEDICAL CENTER,2.0,NaN
3160,PRESBYTERIAN ESPANOLA HOSPITAL,NaN,There are too few measures or measure groups r...
3161,LOVELACE WOMEN'S HOSPITAL,4.0,NaN
3162,MEMORIAL MEDICAL CENTER,2.0,NaN
3163,PRESBYTERIAN HOSPITAL,2.0,NaN


Now that it is clean to add it into the interim folder

In [10]:
print("Rows:", len(hospital_info_nm))
print("Unique hospitals:", hospital_info_nm["Facility ID"].nunique())
print("Available ratings:", hospital_info_nm["overall_rating"].notna().sum())
print("Missing ratings:", hospital_info_nm["overall_rating"].isna().sum())

assert len(hospital_info_nm) == 45
assert hospital_info_nm["Facility ID"].is_unique

hospital_info_nm.to_csv(
    INTERIM / "hospital_information_nm.csv",
    index=False
)

Rows: 45
Unique hospitals: 45
Available ratings: 23
Missing ratings: 22


## 2) Unplanned hospital visits

Each row should represent one hospital and one measure. Missing results remain in the dataset because missingness is part of the coverage analysis.

In [11]:
unplanned_visits = pd.read_csv(
    RAW / "unplanned_hospital_visits.csv",
    dtype=str,
    keep_default_na=False
)

unplanned_visits_nm = unplanned_visits[
    unplanned_visits["State"] == "NM"
].copy()

print("Rows:", len(unplanned_visits_nm))
print("Hospitals:", unplanned_visits_nm["Facility ID"].nunique())
print("Measures:", unplanned_visits_nm["Measure ID"].nunique())

Rows: 574
Hospitals: 41
Measures: 14


### Check the row structure

The expected grain is one row per `Facility ID` and `Measure ID` combination.

In [12]:
duplicate_hospital_measures = unplanned_visits_nm.duplicated(
    subset=["Facility ID", "Measure ID"]
).sum()

print("Duplicate hospital-measure rows:", duplicate_hospital_measures)

assert len(unplanned_visits_nm) == 574
assert unplanned_visits_nm["Facility ID"].nunique() == 41
assert unplanned_visits_nm["Measure ID"].nunique() == 14
assert duplicate_hospital_measures == 0

Duplicate hospital-measure rows: 0


### Create numeric result columns

The original CMS columns remain unchanged. 

The new columns contain numbers, while `Not Available` and `Not Applicable` become `NaN` so they are both consistent with the rest of the data, since they both don't contribute meaningfully.

In [13]:
unplanned_visits_nm["Facility ID"] = (
    unplanned_visits_nm["Facility ID"].str.strip()
)

numeric_columns = {
    "Denominator": "denominator",
    "Score": "score",
    "Lower Estimate": "lower_estimate",
    "Higher Estimate": "higher_estimate",
    "Number of Patients": "number_of_patients",
    "Number of Patients Returned": "number_of_patients_returned",
}

expected_text_values = {"Not Available", "Not Applicable", ""}

for original_column, clean_column in numeric_columns.items():
    unplanned_visits_nm[clean_column] = pd.to_numeric(
        unplanned_visits_nm[original_column],
        errors="coerce"
    )

    failed_values = set(
        unplanned_visits_nm.loc[
            unplanned_visits_nm[clean_column].isna(), original_column
        ].unique()
    )
    unexpected_values = failed_values - expected_text_values
    assert not unexpected_values, (original_column, unexpected_values)

In [14]:
numeric_summary = pd.DataFrame(
    {
        "numeric values": [
            unplanned_visits_nm[column].notna().sum()
            for column in numeric_columns.values()
        ],
        "missing values": [
            unplanned_visits_nm[column].isna().sum()
            for column in numeric_columns.values()
        ],
    },
    index=numeric_columns.values(),
)

display(numeric_summary)

,numeric values,missing values
denominator,236,338
score,236,338
lower_estimate,64,510
higher_estimate,64,510
number_of_patients,68,506
number_of_patients_returned,0,574


### Clean the comparison and decode missing scores

The two versions of `Number of Cases Too Small` differ only in capitalization, so they are standardized. A missing score stays in the row, and its CMS footnote becomes `missing_reason`.

In [15]:
unplanned_visits_nm["comparison_to_national"] = (
    unplanned_visits_nm["Compared to National"]
    .str.strip()
    .replace(
        {
            "Not Available": pd.NA,
            "Number of cases too small": "Number of Cases Too Small",
        }
    )
)

unplanned_visits_nm["missing_reason"] = (
    unplanned_visits_nm["Footnote"]
    .replace("", pd.NA)
    .map(footnote_lookup)
)

unmapped_footnotes = unplanned_visits_nm.loc[
    unplanned_visits_nm["Footnote"].ne("")
    & unplanned_visits_nm["missing_reason"].isna(),
    "Footnote",
].unique()

assert len(unmapped_footnotes) == 0
assert unplanned_visits_nm["score"].isna().equals(
    unplanned_visits_nm["missing_reason"].notna()
)

display(
    unplanned_visits_nm.loc[
        unplanned_visits_nm["score"].isna(),
        ["Footnote", "missing_reason"],
    ].value_counts()
)

Footnote  missing_reason                                                                                                                              
1         The number of cases/patients is too few to report.                                                                                              144
5         Results are not available for this reporting period.                                                                                            139
4         Data suppressed by CMS for one or more quarters.                                                                                                 37
19        Data are shown only for hospitals that participate in the Inpatient Quality Reporting (IQR) and Outpatient Quality Reporting (OQR) programs.     14
7         No cases met the criteria for this measure.                                                                                                       4
Name: count, dtype: int64

### Convert and check reporting dates

Measures use different reporting periods, so the dates must remain attached to each measure.

In [16]:
unplanned_visits_nm["start_date"] = pd.to_datetime(
    unplanned_visits_nm["Start Date"],
    format="%m/%d/%Y"
)

unplanned_visits_nm["end_date"] = pd.to_datetime(
    unplanned_visits_nm["End Date"],
    format="%m/%d/%Y"
)

measure_periods = (
    unplanned_visits_nm[
        ["Measure ID", "Measure Name", "start_date", "end_date"]
    ]
    .drop_duplicates()
    .sort_values("Measure ID")
)

assert len(measure_periods) == 14
assert (measure_periods["start_date"] <= measure_periods["end_date"]).all()

display(measure_periods)

,Measure ID,Measure Name,start_date,end_date
38948,EDAC_30_AMI,Hospital return days for heart attack patients,2022-07-01,2025-06-30
38949,EDAC_30_HF,Hospital return days for heart failure patients,2022-07-01,2025-06-30
38950,EDAC_30_PN,Hospital return days for pneumonia patients,2022-07-01,2025-06-30
38951,Hybrid_HWR,Hybrid Hospital-Wide All-Cause Readmission Mea...,2024-07-01,2025-06-30
38952,OP_32,Rate of unplanned hospital visits after colono...,2022-01-01,2024-12-31
38953,OP_35_ADM,Rate of inpatient admissions for patients rece...,2024-01-01,2024-12-31
38954,OP_35_ED,Rate of emergency department (ED) visits for p...,2024-01-01,2024-12-31
38955,OP_36,Ratio of unplanned hospital visits after hospi...,2024-01-01,2024-12-31
38956,READM_30_AMI,Acute Myocardial Infarction (AMI) 30-Day Readm...,2023-07-01,2025-06-30
38957,READM_30_CABG,Rate of readmission for CABG,2023-07-01,2025-06-30


### Check coverage against the hospital file

This file contains 41 of the 45 New Mexico hospitals. The four absent hospitals are identified here but are not added as artificial measure rows.

In [17]:
unplanned_hospital_ids = set(unplanned_visits_nm["Facility ID"])
hospital_master_ids = set(hospital_info_nm["Facility ID"])

missing_hospitals = hospital_info_nm.loc[
    ~hospital_info_nm["Facility ID"].isin(unplanned_hospital_ids),
    ["Facility ID", "Facility Name", "Hospital Type"],
]

assert unplanned_hospital_ids.issubset(hospital_master_ids)
assert len(missing_hospitals) == 4

print("Hospitals absent from unplanned visits:", len(missing_hospitals))
display(missing_hospitals)

Hospitals absent from unplanned visits: 4


,Facility ID,Facility Name,Hospital Type
3195,324010,"BHC MESILLA VALLEY HOSPITAL, LLC",Psychiatric
3196,324012,"PEAK BEHAVIORAL HEALTH SERVICES, LLC",Psychiatric
3197,324013,HAVEN BEHAVIORAL HOSPITAL OF ALBUQUERQUE,Psychiatric
3198,324014,CENTRAL DESERT BEHAVIORAL HEALTH HOSPITAL,Psychiatric


### Important interpretation

`score` is numeric, but it does not have one shared meaning across all 14 measures. It includes return days, rates and ratios. Compare scores only within the same `Measure ID`.

### Validate and save

In [18]:
assert len(unplanned_visits_nm) == 574
assert not unplanned_visits_nm.duplicated(
    subset=["Facility ID", "Measure ID"]
).any()
assert unplanned_visits_nm["score"].notna().sum() == 236
assert unplanned_visits_nm["score"].isna().sum() == 338
assert unplanned_visits_nm["score"].notna().equals(
    unplanned_visits_nm["denominator"].notna()
)

unplanned_visits_nm.to_csv(
    INTERIM / "unplanned_visits_nm.csv",
    index=False
)

print("Saved:", INTERIM / "unplanned_visits_nm.csv")

Saved: ../data/interim/unplanned_visits_nm.csv


## 3) Complication and deaths

In [19]:
complications_death = pd.read_csv(
    RAW / "complications_and_deaths.csv",
    dtype=str,
    keep_default_na=False,
)

complications_death_nm = complications_death[
    complications_death["State"] == "NM"
].copy()

print("Rows:", len(complications_death_nm))
print("Hospitals:", complications_death_nm["Facility ID"].nunique())
print("Measures:", complications_death_nm["Measure ID"].nunique())


Rows: 820
Hospitals: 41
Measures: 20


### Check the row structure

The expected grain is one row per `Facility ID` and `Measure ID` combination.
820 rows = 41 hospitals x 20 measures.


In [20]:
complications_death_nm["Facility ID"] = (
    complications_death_nm["Facility ID"].str.strip()
)

duplicate_hospital_measures = complications_death_nm.duplicated(
    subset=["Facility ID", "Measure ID"]
).sum()

print("Duplicate hospital-measure rows:", duplicate_hospital_measures)

assert len(complications_death_nm) == 820
assert complications_death_nm["Facility ID"].nunique() == 41
assert complications_death_nm["Measure ID"].nunique() == 20
assert duplicate_hospital_measures == 0


Duplicate hospital-measure rows: 0


### Create numeric result columns

This file has no patient-count columns, so only four columns are converted.
The original CMS columns remain unchanged. The new columns contain numbers;
`Not Available` and `Not Applicable` both coerce to `NaN`. Here, `Not Applicable`
appears only on the `PSI_90` denominator (a composite index that has no single
denominator). `Not Available` means CMS suppressed the value; the reason is kept
separately in `missing_reason`.


In [21]:
numeric_columns = {
    "Denominator": "denominator",
    "Score": "score",
    "Lower Estimate": "lower_estimate",
    "Higher Estimate": "higher_estimate",
}

expected_text_values = {"Not Available", "Not Applicable", ""}

for original_column, clean_column in numeric_columns.items():
    complications_death_nm[clean_column] = pd.to_numeric(
        complications_death_nm[original_column],
        errors="coerce"
    )

    failed_values = set(
        complications_death_nm.loc[
            complications_death_nm[clean_column].isna(), original_column
        ].unique()
    )
    unexpected_values = failed_values - expected_text_values
    assert not unexpected_values, (original_column, unexpected_values)


In [22]:
numeric_summary = pd.DataFrame(
    {
        "numeric values": [
            complications_death_nm[column].notna().sum()
            for column in numeric_columns.values()
        ],
        "missing values": [
            complications_death_nm[column].isna().sum()
            for column in numeric_columns.values()
        ],
    },
    index=numeric_columns.values(),
)

display(numeric_summary)


,numeric values,missing values
denominator,364,456
score,388,432
lower_estimate,239,581
higher_estimate,239,581


### Clean the comparison and decode missing scores

`Compared to National` uses two vocabularies: "... the National Rate" (complication
and safety rate measures) and "... the National Value" (mortality measures). Both are
kept as-is here and unified later in the scorecard. "Not Available" becomes missing.

A missing score keeps its row, and its CMS footnote becomes `missing_reason`. In this
file a score can exist while the national comparison is "Not Available" (CMS reports a
risk-standardized rate but makes no better/worse determination), so comparison
missingness does **not** equal score missingness here. What matches the missing score
exactly is the footnote: every suppressed score has a footnote, and every scored row
has none.


In [23]:
complications_death_nm["comparison_to_national"] = (
    complications_death_nm["Compared to National"]
    .str.strip()
    .replace({"Not Available": pd.NA})
)

complications_death_nm["missing_reason"] = (
    complications_death_nm["Footnote"]
    .replace("", pd.NA)
    .map(footnote_lookup)
)

unmapped_footnotes = complications_death_nm.loc[
    complications_death_nm["Footnote"].ne("")
    & complications_death_nm["missing_reason"].isna(),
    "Footnote",
].unique()

assert len(unmapped_footnotes) == 0
assert complications_death_nm["score"].isna().equals(
    complications_death_nm["missing_reason"].notna()
)

display(
    complications_death_nm.loc[
        complications_death_nm["score"].isna(),
        ["Footnote", "missing_reason"],
    ].value_counts()
)


Footnote  missing_reason                                                                                                                              
1         The number of cases/patients is too few to report.                                                                                              144
13        Results cannot be calculated for this reporting period.                                                                                         132
5         Results are not available for this reporting period.                                                                                            112
7         No cases met the criteria for this measure.                                                                                                      24
19        Data are shown only for hospitals that participate in the Inpatient Quality Reporting (IQR) and Outpatient Quality Reporting (OQR) programs.     20
Name: count, dtype: int64

### Convert and check reporting dates

Measures use different reporting periods, so the dates must remain attached to each
measure.


In [24]:
complications_death_nm["start_date"] = pd.to_datetime(
    complications_death_nm["Start Date"],
    format="%m/%d/%Y"
)

complications_death_nm["end_date"] = pd.to_datetime(
    complications_death_nm["End Date"],
    format="%m/%d/%Y"
)

measure_periods = (
    complications_death_nm[
        ["Measure ID", "Measure Name", "start_date", "end_date"]
    ]
    .drop_duplicates()
    .sort_values("Measure ID")
)

assert len(measure_periods) == 20
assert (measure_periods["start_date"] <= measure_periods["end_date"]).all()

display(measure_periods)


,Measure ID,Measure Name,start_date,end_date
55640,COMP_HIP_KNEE,Rate of complications for hip/knee replacement...,2023-04-01,2025-03-31
55641,Hybrid_HWM,Hybrid Hospital-Wide All-Cause Risk Standardiz...,2024-07-01,2025-06-30
55642,MORT_30_AMI,Death rate for heart attack patients,2022-07-01,2025-06-30
55643,MORT_30_CABG,Death rate for CABG surgery patients,2022-07-01,2025-06-30
55644,MORT_30_COPD,Death rate for COPD patients,2022-07-01,2025-06-30
55645,MORT_30_HF,Death rate for heart failure patients,2022-07-01,2025-06-30
55646,MORT_30_PN,Death rate for pneumonia patients,2022-07-01,2025-06-30
55647,MORT_30_STK,Death rate for stroke patients,2023-07-01,2025-06-30
55648,PSI_03,Pressure ulcer rate,2022-07-01,2024-06-30
55649,PSI_04,Death rate among surgical inpatients with seri...,2022-07-01,2024-06-30


### Check coverage against the hospital file

This file contains the same 41 of 45 New Mexico hospitals as the readmissions file.
The four absent hospitals are identified here but are not added as artificial measure
rows.


In [25]:
complications_hospital_ids = set(complications_death_nm["Facility ID"])
hospital_master_ids = set(hospital_info_nm["Facility ID"])

missing_hospitals = hospital_info_nm.loc[
    ~hospital_info_nm["Facility ID"].isin(complications_hospital_ids),
    ["Facility ID", "Facility Name", "Hospital Type"],
]

assert complications_hospital_ids.issubset(hospital_master_ids)
assert len(missing_hospitals) == 4

print("Hospitals absent from complications and deaths:", len(missing_hospitals))
display(missing_hospitals)


Hospitals absent from complications and deaths: 4


,Facility ID,Facility Name,Hospital Type
3195,324010,"BHC MESILLA VALLEY HOSPITAL, LLC",Psychiatric
3196,324012,"PEAK BEHAVIORAL HEALTH SERVICES, LLC",Psychiatric
3197,324013,HAVEN BEHAVIORAL HOSPITAL OF ALBUQUERQUE,Psychiatric
3198,324014,CENTRAL DESERT BEHAVIORAL HEALTH HOSPITAL,Psychiatric


### Important interpretation

`score` is numeric, but it does not have one shared meaning across all 20 measures. It
mixes death rates, complication rates, patient-safety (PSI) rates and the PSI_90
composite index. Compare scores only within the same `Measure ID`. `PSI_90` has no
denominator (composite), so a present score with a missing denominator is expected for
that measure only.


### Validate and save


In [26]:
assert len(complications_death_nm) == 820
assert not complications_death_nm.duplicated(
    subset=["Facility ID", "Measure ID"]
).any()
assert complications_death_nm["score"].notna().sum() == 388
assert complications_death_nm["score"].isna().sum() == 432
assert complications_death_nm["score"].isna().equals(
    complications_death_nm["missing_reason"].notna()
)

# denominator is numeric wherever a score exists, except PSI_90 (composite, no denominator)
denominator_missing_with_score = (
    complications_death_nm["denominator"].isna()
    & complications_death_nm["score"].notna()
)
assert set(
    complications_death_nm.loc[denominator_missing_with_score, "Measure ID"]
) == {"PSI_90"}

complications_death_nm.to_csv(
    INTERIM / "complications_death_nm.csv",
    index=False
)

print("Saved:", INTERIM / "complications_death_nm.csv")


Saved: ../data/interim/complications_death_nm.csv


## 4) HCAHPS

In [27]:
hcahps = pd.read_csv(
    RAW / "hcahps_patient_survey.csv",
    dtype=str,
    keep_default_na=False,
)

hcahps_nm = hcahps[hcahps["State"] == "NM"].copy()

print("Rows:", len(hcahps_nm))
print("Hospitals:", hcahps_nm["Facility ID"].nunique())
print("Measures:", hcahps_nm["HCAHPS Measure ID"].nunique())


Rows: 2788
Hospitals: 41
Measures: 68


### Check the row structure

The expected grain is one row per `Facility ID` and `HCAHPS Measure ID` combination.
2,788 rows = 41 hospitals x 68 measures.


In [28]:
hcahps_nm["Facility ID"] = hcahps_nm["Facility ID"].str.strip()

duplicate_hospital_measures = hcahps_nm.duplicated(
    subset=["Facility ID", "HCAHPS Measure ID"]
).sum()

print("Duplicate hospital-measure rows:", duplicate_hospital_measures)

assert len(hcahps_nm) == 2788
assert hcahps_nm["Facility ID"].nunique() == 41
assert hcahps_nm["HCAHPS Measure ID"].nunique() == 68
assert duplicate_hospital_measures == 0


Duplicate hospital-measure rows: 0


### Label each row by measure kind

HCAHPS is stored long: each row is one of three kinds, and only one value column is
meant to be filled. The suffix of `HCAHPS Measure ID` tells us which:
- `_STAR_RATING` -> a 1-5 star
- `_LINEAR_SCORE` -> a linear mean score
- everything else -> an answer percent

`measure_kind` records this so we know which column holds the real value for each row.


In [29]:
def measure_kind(measure_id):
    if measure_id.endswith("_STAR_RATING"):
        return "star"
    if measure_id.endswith("_LINEAR_SCORE"):
        return "linear"
    return "answer_pct"

hcahps_nm["measure_kind"] = hcahps_nm["HCAHPS Measure ID"].map(measure_kind)

print(hcahps_nm["measure_kind"].value_counts())


measure_kind
answer_pct    2091
star           369
linear         328
Name: count, dtype: int64


### Build one value column per row

Each kind has one "primary" column. We pull that into a single numeric `value` column
(the HCAHPS equivalent of `score` in the other files). Because we only take each row's
own column, "Not Applicable" (which only appears in the columns that do not apply to a
row) never lands in `value` - it is either a number or "Not Available".

`completed_surveys` and `response_rate` are survey-volume context that apply to every
row, so they are converted too.


In [30]:
primary_column = {
    "star": "Patient Survey Star Rating",
    "linear": "HCAHPS Linear Mean Value",
    "answer_pct": "HCAHPS Answer Percent",
}

# pick each row's primary raw value based on its kind
primary_raw = hcahps_nm.apply(
    lambda row: row[primary_column[row["measure_kind"]]],
    axis=1,
)

hcahps_nm["value"] = pd.to_numeric(primary_raw, errors="coerce")

# survey-volume context (applies to every row)
hcahps_nm["completed_surveys"] = pd.to_numeric(
    hcahps_nm["Number of Completed Surveys"], errors="coerce"
)
hcahps_nm["response_rate"] = pd.to_numeric(
    hcahps_nm["Survey Response Rate Percent"], errors="coerce"
)

# the only non-numeric primary values should be "Not Available" (no "Not Applicable")
failed_primary = set(primary_raw[hcahps_nm["value"].isna()].unique())
assert failed_primary <= {"Not Available", ""}, failed_primary

print("Rows with a value:", hcahps_nm["value"].notna().sum())
print("Rows missing a value:", hcahps_nm["value"].isna().sum())


Rows with a value: 2041
Rows missing a value: 747


### Decode why a value is missing (primary value only)

For the suppressed values ("Not Available"), decode the footnote that belongs to that
row's primary column: star rows use the star footnote, answer-percent rows use the
answer-percent footnote. Linear rows have NO footnote column in the CMS file, so a
suppressed linear value has no decodable reason - a known exception, handled like
PSI_90 in the complications file.


In [31]:
primary_footnote_column = {
    "star": "Patient Survey Star Rating Footnote",
    "answer_pct": "HCAHPS Answer Percent Footnote",
    "linear": None,  # CMS provides no footnote column for linear scores
}

def primary_footnote(row):
    column = primary_footnote_column[row["measure_kind"]]
    if column is None:
        return ""
    return row[column]

footnote_code = hcahps_nm.apply(primary_footnote, axis=1).replace("", pd.NA)
hcahps_nm["missing_reason"] = footnote_code.map(footnote_lookup)

# every non-blank footnote must decode
unmapped_footnotes = footnote_code[
    footnote_code.notna() & hcahps_nm["missing_reason"].isna()
].unique()
assert len(unmapped_footnotes) == 0, unmapped_footnotes

# a missing value with no reason should only be a linear row (no footnote column)
missing_without_reason = (
    hcahps_nm["value"].isna() & hcahps_nm["missing_reason"].isna()
)
assert set(hcahps_nm.loc[missing_without_reason, "measure_kind"]) <= {"linear"}

display(
    hcahps_nm.loc[
        hcahps_nm["value"].isna(),
        ["measure_kind", "missing_reason"],
    ].value_counts(dropna=False)
)


measure_kind  missing_reason                                                                                                                              
answer_pct    Results are not available for this reporting period.                                                                                            255
              The number of cases/patients is too few to report.                                                                                              186
linear        NaN                                                                                                                                             120
star          The number of cases/patients is too few to report a star rating.                                                                                 81
answer_pct    Data are shown only for hospitals that participate in the Inpatient Quality Reporting (IQR) and Outpatient Quality Reporting (OQR) programs.     51
star          Results are not avail

### Convert reporting dates

All HCAHPS rows share one reporting period, but convert and confirm it.


In [32]:
hcahps_nm["start_date"] = pd.to_datetime(hcahps_nm["Start Date"], format="%m/%d/%Y")
hcahps_nm["end_date"] = pd.to_datetime(hcahps_nm["End Date"], format="%m/%d/%Y")

periods = hcahps_nm[["start_date", "end_date"]].drop_duplicates()

assert len(periods) == 1
assert (hcahps_nm["start_date"] <= hcahps_nm["end_date"]).all()

display(periods)


,start_date,end_date
189176,2024-10-01,2025-09-30


### Check coverage against the hospital file

Like the other measure files, HCAHPS covers 41 of 45 NM hospitals; the same four
Psychiatric hospitals are absent.


In [33]:
hcahps_hospital_ids = set(hcahps_nm["Facility ID"])
hospital_master_ids = set(hospital_info_nm["Facility ID"])

missing_hospitals = hospital_info_nm.loc[
    ~hospital_info_nm["Facility ID"].isin(hcahps_hospital_ids),
    ["Facility ID", "Facility Name", "Hospital Type"],
]

assert hcahps_hospital_ids.issubset(hospital_master_ids)
assert len(missing_hospitals) == 4

print("Hospitals absent from HCAHPS:", len(missing_hospitals))
display(missing_hospitals)


Hospitals absent from HCAHPS: 4


,Facility ID,Facility Name,Hospital Type
3195,324010,"BHC MESILLA VALLEY HOSPITAL, LLC",Psychiatric
3196,324012,"PEAK BEHAVIORAL HEALTH SERVICES, LLC",Psychiatric
3197,324013,HAVEN BEHAVIORAL HOSPITAL OF ALBUQUERQUE,Psychiatric
3198,324014,CENTRAL DESERT BEHAVIORAL HEALTH HOSPITAL,Psychiatric


### Important interpretation

`value` means different things by `measure_kind` (a 1-5 star, a 0-100 percent, or a
linear mean), so compare values only within the same `HCAHPS Measure ID`. The headline
patient-experience number for the scorecard is the summary star, `H_STAR_RATING`
(one row per hospital); 26 of 45 NM hospitals have a usable value for it.


### Validate and save


In [34]:
assert len(hcahps_nm) == 2788
assert not hcahps_nm.duplicated(subset=["Facility ID", "HCAHPS Measure ID"]).any()
assert hcahps_nm["value"].notna().sum() == 2041
assert hcahps_nm["value"].isna().sum() == 747

hcahps_nm.to_csv(INTERIM / "hcahps_nm.csv", index=False)

print("Saved:", INTERIM / "hcahps_nm.csv")


Saved: ../data/interim/hcahps_nm.csv


## 5) RUCA

In [35]:
ruca = pd.read_csv(
    RAW / "ruca_zip_2010.csv",
    dtype=str,
    keep_default_na=False,
)

# the ZIP column name AND its values are wrapped in doubled quotes: ''00001''
zip_column = "''ZIP_CODE''"
ruca["zip"] = ruca[zip_column].str.replace("'", "", regex=False).str[:5]

print("RUCA rows:", len(ruca))
print("sample cleaned zips:", ruca["zip"].head(3).tolist())


RUCA rows: 41164
sample cleaned zips: ['00001', '00002', '00003']


### Filter to New Mexico

RUCA is national; keep the NM ZIPs. Each ZIP should appear once (a clean 1-to-1 lookup).


In [36]:
ruca_nm = ruca[ruca["STATE"] == "NM"].copy()

print("NM ZIPs:", len(ruca_nm))
print("duplicate ZIPs:", ruca_nm["zip"].duplicated().sum())

assert ruca_nm["zip"].is_unique


NM ZIPs: 429
duplicate ZIPs: 0


### Group RUCA codes into rural/urban

`RUCA1` is a 1-10 code. USDA's standard grouping:
- 1-3 metropolitan
- 4-6 micropolitan
- 7-9 small town
- 10 rural


In [37]:
def rural_urban(ruca1):
    code = int(ruca1)
    if code <= 3:
        return "metropolitan"
    if code <= 6:
        return "micropolitan"
    if code <= 9:
        return "small town"
    return "rural"

ruca_nm["rural_urban"] = ruca_nm["RUCA1"].map(rural_urban)

print(ruca_nm["rural_urban"].value_counts())


rural_urban
metropolitan    168
rural           124
micropolitan     87
small town       50
Name: count, dtype: int64


### Join rural/urban onto the hospital table

Add each hospital's rural/urban group by its 5-digit ZIP (already cleaned to 5 chars in
section 1). Every NM hospital ZIP should match a RUCA record.


In [38]:
ruca_lookup = ruca_nm.set_index("zip")[["RUCA1", "rural_urban"]]

hospital_info_nm["ruca1"] = hospital_info_nm["ZIP Code"].map(ruca_lookup["RUCA1"])
hospital_info_nm["rural_urban"] = hospital_info_nm["ZIP Code"].map(ruca_lookup["rural_urban"])

matched = hospital_info_nm["rural_urban"].notna().sum()
print("hospital ZIPs matched to RUCA:", matched, "/", len(hospital_info_nm))

assert hospital_info_nm["rural_urban"].notna().all()

display(hospital_info_nm[["Facility Name", "ZIP Code", "ruca1", "rural_urban"]].head())


hospital ZIPs matched to RUCA: 45 / 45


,Facility Name,ZIP Code,ruca1,rural_urban
3154,UNM HOSPITAL,87106,1,metropolitan
3155,CHRISTUS ST VINCENT REGIONAL MEDICAL CENTER,87505,1,metropolitan
3156,CHRISTUS SOUTHERN NEW MEXICO,88310,4,micropolitan
3157,SAN JUAN REGIONAL MEDICAL CENTER INC,87401,1,metropolitan
3158,EASTERN NEW MEXICO MEDICAL CENTER,88201,4,micropolitan


### NM hospitals by rural/urban

A first look at the coverage story (question 2): how the 45 NM hospitals split.


In [39]:
print(hospital_info_nm["rural_urban"].value_counts())


rural_urban
metropolitan    18
micropolitan    16
small town       9
rural            2
Name: count, dtype: int64


## Validation

Cross-file checks before building the analysis-ready tables: every hospital in a
measure file exists in the hospital spine, and the hospital table is one row per
hospital.


In [40]:
hospital_ids = set(hospital_info_nm["Facility ID"])

for name, frame in [
    ("unplanned", unplanned_visits_nm),
    ("complications", complications_death_nm),
    ("hcahps", hcahps_nm),
]:
    extra = set(frame["Facility ID"]) - hospital_ids
    assert not extra, (name, extra)
    print(f"{name}: all Facility IDs present in hospital table")

assert hospital_info_nm["Facility ID"].is_unique
assert len(hospital_info_nm) == 45


unplanned: all Facility IDs present in hospital table
complications: all Facility IDs present in hospital table
hcahps: all Facility IDs present in hospital table


## Save cleaned data

Two analysis-ready tables in `data/processed/`:
- `hospitals.csv` - one row per hospital (spine: attributes, rating, rural/urban, and which measure sets cover it).
- `hospital_measures.csv` - one row per hospital-measure across all three sources, with hospital attributes joined on.


In [41]:
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(exist_ok=True)

# which measure sets include each hospital (coverage flags for question 2)
readm_ids = set(unplanned_visits_nm["Facility ID"])
comp_ids = set(complications_death_nm["Facility ID"])
hcahps_ids = set(hcahps_nm["Facility ID"])

hospitals = pd.DataFrame({
    "facility_id": hospital_info_nm["Facility ID"],
    "facility_name": hospital_info_nm["Facility Name"],
    "city": hospital_info_nm["City/Town"],
    "county": hospital_info_nm["County/Parish"],
    "zip": hospital_info_nm["ZIP Code"],
    "hospital_type": hospital_info_nm["Hospital Type"],
    "ownership": hospital_info_nm["Hospital Ownership"],
    "is_critical_access": hospital_info_nm["is_critical_access"],
    "overall_rating": hospital_info_nm["overall_rating"],
    "overall_rating_missing_reason": hospital_info_nm["overall_rating_missing_reason"],
    "ruca1": hospital_info_nm["ruca1"],
    "rural_urban": hospital_info_nm["rural_urban"],
})
hospitals["in_readmission"] = hospitals["facility_id"].isin(readm_ids)
hospitals["in_complications"] = hospitals["facility_id"].isin(comp_ids)
hospitals["in_hcahps"] = hospitals["facility_id"].isin(hcahps_ids)

assert len(hospitals) == 45
assert hospitals["facility_id"].is_unique

hospitals.to_csv(PROCESSED / "hospitals.csv", index=False)
print("Saved:", PROCESSED / "hospitals.csv", "|", len(hospitals), "rows")


Saved: ../data/processed/hospitals.csv | 45 rows


### Unified hospital-measure table

One tidy row per hospital-measure across all three sources. `value` is the
comparable-within-measure number (readmission/complication `score`, or HCAHPS `value`).
`reported` flags whether a value exists - the backbone of the coverage analysis.
`denominator` and `comparison_to_national` apply to the readmission/complication
sources only (blank for patient experience).


In [42]:
def tidy(frame, source, measure_id_col, measure_name_col, value_col,
         denom_col=None, comparison_col=None):
    return pd.DataFrame({
        "facility_id": frame["Facility ID"],
        "source": source,
        "measure_id": frame[measure_id_col],
        "measure_name": frame[measure_name_col],
        "value": frame[value_col],
        "denominator": frame[denom_col] if denom_col else pd.NA,
        "comparison_to_national": frame[comparison_col] if comparison_col else pd.NA,
        "missing_reason": frame["missing_reason"],
        "start_date": frame["start_date"],
        "end_date": frame["end_date"],
    })

measures = pd.concat([
    tidy(unplanned_visits_nm, "readmission", "Measure ID", "Measure Name",
         "score", "denominator", "comparison_to_national"),
    tidy(complications_death_nm, "complications", "Measure ID", "Measure Name",
         "score", "denominator", "comparison_to_national"),
    tidy(hcahps_nm, "patient_experience", "HCAHPS Measure ID", "HCAHPS Question",
         "value"),
], ignore_index=True)

measures["reported"] = measures["value"].notna()

# join the hospital attributes the coverage/funnel analysis needs
attrs = hospitals.set_index("facility_id")[
    ["hospital_type", "is_critical_access", "rural_urban"]
]
measures = measures.join(attrs, on="facility_id")

assert len(measures) == 574 + 820 + 2788
assert measures["hospital_type"].notna().all()

measures.to_csv(PROCESSED / "hospital_measures.csv", index=False)
print("Saved:", PROCESSED / "hospital_measures.csv", "|", len(measures), "rows")
print(measures["source"].value_counts())


Saved: ../data/processed/hospital_measures.csv | 4182 rows
source
patient_experience    2788
complications          820
readmission            574
Name: count, dtype: int64


/var/folders/b3/6y8qhvt56_z1swbjbctplhcc0000gn/T/ipykernel_64810/1455941049.py:16: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  measures = pd.concat([
